# CaloChallenge 2022 - Dataset 2
> Harrison B. Prosper, January 2026

## Introduction

The goal of the [Fast Calorimeter Simulation Challenge](https://calochallenge.github.io/homepage/) (CaloChallenge) is to encourage the development of fast, accurate, simulators of the particle showers created in a voxelized calorimeter when a particle interacts with it. (A voxel is the 3D analog of a pixel.) The CaloChallenge team provides three datasets of simulated showers. This notebook plots showers from `Dataset 2`. For convenience of ML applications with PyTorch, the data have been reformated and stored in a PyTorch data file: `data/Dataset2.pt`. 

The graphic below shows the layout of the simulated calorimeter. Note: In this notebook, we use the symbol $
\theta$ for the angle labeled $\alpha$ in the figure. 

<img src="./coordsys.jpg" align="left" width="500px"/>

The simulated calorimeter consists of several layers of detectors, each of which is divided into cells (i.e., voxels). For Dataset 2, the voxelization of the simulated calorimeter is defined by $(L, A, R)$, where $L = 45$ is the number of layers along the $z$-axis, i.e., the incident particle direction, $A = 16$ is the number of angular bins, $\theta$ (labeled $\alpha$ in the figure), and $R=9$ is the number of radial bins, $r$. Therefore, each shower in Dataset 2 is defined by $45
\times 16 
\times 9 = 6480$ voxels.

The simulated detector has cylindrical symmetry. 

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import PowerNorm
import torch
import torch.nn as nn

## Load Data
Load the data in memory-mapped mode so that the data are loaded only when they are needed. This uses less memory than loading all the data at once.

Note: For each `shower`, the energy of the incident particle is given in the tensor `energies`.

In [ ]:
data = torch.load('data/Dataset2.pt', mmap=True)
energies = data['energies'] / 1000 # incident particle energies in GeV
showers  = data['showers']

N_SHOWERS, L, A, R = showers.shape
N_VOXELS = L * A * R

print(f'number of showers: {N_SHOWERS:10d}')
print(f'number of voxels:  {N_VOXELS:10d}')

energies.shape, showers.shape

In [ ]:
energies.min(), energies.max()

## Define Plots
  * X-Y plot: Plot the data with $x$-axis the angular coordinate, and the $y$-axis the radial coordinate.
  * Polar plot: Plot the data in polar coordinates.

In [ ]:
def plot_shower_xy(data, 
                   n=0,
                   P=0.3,
                   ftsize=12, 
                   filename='shower_xy_NNN.png'):

    filename = filename.replace('NNN', '%6.6d' % n)

    shower  = data[n]
    L, A, R = shower.shape

    # -------------------------
    # Figure + axes
    # -------------------------
    ncols = 5 # number of columns
    nrows = int(np.ceil(L / ncols)) # number of rows

    fig, axes = plt.subplots(
        nrows, ncols,
        figsize=(2.0*ncols, 1.3*nrows),
        constrained_layout=True     # provide reasonable layout for subplots
    )

    # -------------------------
    # Meshgrid of edges
    # ------------------------- 
    # Change axes to a flat 1D NumPy array of Axes objects.
    # Use ravel() to force row-wise flattening.
    axes = np.atleast_1d(axes).ravel()

    # Compute centers of each bin in (x, y)-plane
    x = np.linspace(0, A-1, A) + 0.5
    y = np.linspace(0, R-1, R) + 0.5
    x, y = np.meshgrid(x, y, indexing="ij")
    x = x.flatten()
    y = y.flatten()

    xmin, xmax = 0, A
    ymin, ymax = 0, R
    
    # loop over layers
    for ax, idx in zip(axes, range(L)):
        
        layer = shower[idx]
        
        # Hide labels & ticks
        ax.set_xticks([])
        ax.set_yticks([])
        ax.set_xticklabels([])
        ax.set_yticklabels([])
        ax.tick_params(axis='both', which='both', length=0, labelsize=0)
    
        ax.set_title(f'Layer {idx}', fontsize=ftsize)

        # Create a 2D histogram where the histogram bin weights are 
        # the energies in the calorimeter cells (voxels).
        energies = layer.flatten()
        ax.hist2d(x, y, 
                  bins=[A, R], 
                  range=[(xmin, xmax), (ymin, ymax)], 
                  norm=PowerNorm(gamma=P),
                  cmap='viridis',
                  weights=energies)

    plt.savefig(filename)
    plt.show()

In [ ]:
def plot_shower_polar(data,
                      n=0,                 
                      P=0.3,
                      ftsize=12, 
                      filename='shower_polar_NNN.png'):

    filename = filename.replace('NNN', '%6.6d' % n)
        
    shower = data[n]
    L, A, R = shower.shape

    # -------------------------
    # Bin edges
    # -------------------------
    theta_edges = np.linspace(0, 2*np.pi, A + 1)
    r_edges     = np.linspace(0.0, 1.0, R + 1)

    # -------------------------
    # Meshgrid of edges
    # -------------------------
    theta, r = np.meshgrid(theta_edges, r_edges, indexing="ij")
    
    # -------------------------
    # Figure + axes
    # -------------------------
    ncols = 5
    nrows = int(np.ceil(L / ncols))
    
    fig, axes = plt.subplots(
        nrows, ncols,
        subplot_kw={'projection': 'polar'}, #dict(projection="polar"),
        figsize=(2.0*ncols, 2.0*nrows),
        constrained_layout=True
    )

    # Force axes to become a flat 1D NumPy array 
    # of Axes objects and use ravel() to force 
    # row-wise flattening.
    axes = np.atleast_1d(axes).ravel()

    # loop over layers
    for ax, idx in zip(axes, range(L)):
        
        energies = shower[idx]
        
        pcm = ax.pcolormesh(
            theta, r, energies,
            shading="auto",  # figure out a consistent shading
            norm=PowerNorm(gamma=P),
            cmap="viridis"
        )

        ax.set_title(f"Layer {idx}", pad=8, fontsize=ftsize)
        ax.set_theta_zero_location("E") # pointing "Eastwards"
        ax.set_theta_direction(-1)      # counter-clockwise

        # Hide labels & ticks
        ax.set_xticks([])
        ax.set_yticks([])
        ax.set_xticklabels([])
        ax.set_yticklabels([])
        ax.tick_params(axis='both', which='both', length=0, labelsize=0)

        # Draw bin boundaries
        ax.set_thetagrids(np.degrees(theta_edges))
        ax.set_rgrids(r_edges)
        ax.grid(True, color="white", linewidth=0.6, alpha=0.7)

    plt.savefig(filename)
    plt.show()

In [ ]:
IDX = 0

plot_shower_polar(showers, n=IDX)

print('\nradial (x) vs angular (y) bins\n')

plot_shower_xy(showers, n=IDX)